In [1]:

# MULTILINGUAL SENTIMENT ANALYSIS WITH FINE-TUNING
# Dataset: NaijaSenti
# Pretrained Model: cardiffnlp/twitter-xlm-roberta-base-sentiment

# STEP 1: IMPORT LIBRARIES

# These libraries help with: data loading, model loading, training and evaluation

import pandas as pd, zipfile, os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from datasets import Dataset


In [2]:

# STEP 2: LOAD DATASET

# Clone the NaijaSenti repository
if not os.path.exists("NaijaSenti"):
    !git clone https://github.com/hausanlp/NaijaSenti.git

# Extract the data.zip file
with zipfile.ZipFile("NaijaSenti/data.zip", 'r') as zip_ref:
    zip_ref.extractall("NaijaSenti")

# Load all 4 languages and combine them
languages = ["hau", "ibo", "pcm", "yor"]
all_data = []

for lang in languages:
    df = pd.read_csv(f"NaijaSenti/data/annotated_tweets/{lang}/train.tsv", sep='\t')
    print(f"{lang.upper()} Samples: {len(df)}")
    all_data.append(df)

data = pd.concat(all_data, ignore_index=True)

# Keep only text and label columns
data = data[["tweet", "label"]].rename(columns={"tweet": "text"})

# Convert labels to numbers: negative=0, neutral=1, positive=2
data["label"] = data["label"].map({"negative": 0, "neutral": 1, "positive": 2})

# Remove empty rows
data = data.dropna()

print(f"\nTotal Samples (All Languages): {len(data)}")

Cloning into 'NaijaSenti'...
remote: Enumerating objects: 1533, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 1533 (delta 98), reused 182 (delta 61), pack-reused 1305 (from 1)
Receiving objects: 100% (1533/1533), 30.18 MiB | 14.40 MiB/s, done.
Resolving deltas: 100% (956/956), done.
HAU Samples: 14172
IBO Samples: 10192
PCM Samples: 5121
YOR Samples: 8522

Total Samples (All Languages): 38007


In [3]:
# STEP 3: DATA PREPROCESSING
# Remove empty rows
data = data.dropna()
data["text"] = data["text"].astype(str)

# Check what labels look like
print("Unique labels found:", data["label"].unique())
print("Label data type:", data["label"].dtype)

Unique labels found: [0 1 2]
Label data type: int64


In [4]:
# STEP 4: DATA Splitting
# Training set = 80% of data (used to train the model)
# Testing set  = 20% of data (used to evaluate the model)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    data["text"],
    data["label"],
    test_size=0.2,      # 20% for testing
    train_size=0.8,     # 80% for training
    random_state=1      # ensures same split every time you run
)

print("Data Splitting Completed")
print(f"Total Samples:    {len(data)}")
print(f"Training Samples: {len(train_texts)} (80%)")
print(f"Testing Samples:  {len(test_texts)} (20%)")

Data Splitting Completed
Total Samples:    38007
Training Samples: 30405 (80%)
Testing Samples:  7602 (20%)


In [5]:
# STEP 5: LOAD PRETRAINED MODEL AND TOKENIZER

model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment" # pretrained multilingual sentiment model
tokenizer = AutoTokenizer.from_pretrained(model_name)         # tokenizer converts text to numbers the model understands
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3) # load model with 3 output labels (negative, neutral, positive)

print("Pretrained Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pretrained Model Loaded


In [6]:
# STEP 6: TOKENIZATION

# tokenize function: converts text to numbers the model understands
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

# convert pandas dataframes to HuggingFace dataset format
train_dataset = Dataset.from_dict({"text": train_texts.tolist(), "label": train_labels.tolist()})
test_dataset = Dataset.from_dict({"text": test_texts.tolist(), "label": test_labels.tolist()})

# apply tokenization to both datasets
train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

print("Tokenization Completed")

Map:   0%|          | 0/30405 [00:00<?, ? examples/s]

Map:   0%|          | 0/7602 [00:00<?, ? examples/s]

Tokenization Completed


In step 6, we are tokenizing the **NaijaSenti dataset** using the **pretrained model's tokenizer**.

- The **NaijaSenti dataset** contains raw text (tweets in Hausa, Igbo, Pidgin, Yoruba)
- The **pretrained model's tokenizer** is the tool we use to convert that raw text into numbers
- The model can only understand numbers, not raw text

So in simple terms — we are **preparing the NaijaSenti tweets** in a format that the **twitter-xlm-roberta model** can understand and learn from.

In [7]:
# STEP 7: TRAINING SETTINGS and Evaluation Function

from transformers import TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import torch

# TRAINING SETTINGS
training_args = TrainingArguments(
    output_dir="./results",        # folder to save the trained model
    num_train_epochs=1,            # number of training cycles
    per_device_train_batch_size=16, # number of samples processed at a time during training
    per_device_eval_batch_size=8,  # number of samples processed at a time during evaluation
    eval_strategy="epoch",         # evaluate after each epoch
    save_strategy="epoch",         # save model after each epoch
    logging_dir="./logs",          # folder to save training logs
)

print("Training Settings Configured")

# EVALUATION FUNCTION
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=1)  # pick highest scoring label
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Settings Configured


In [8]:
# STEP 8: CREATE TRAINER AND TRAIN MODEL

# Fix: remove NaN labels and convert to integers
train_dataset = train_dataset.filter(lambda x: x["label"] is not None and str(x["label"]) != "nan")
test_dataset = test_dataset.filter(lambda x: x["label"] is not None and str(x["label"]) != "nan")

# Fix: convert labels from Float to Long (integers)
train_dataset = train_dataset.map(lambda x: {"label": int(x["label"])})
test_dataset = test_dataset.map(lambda x: {"label": int(x["label"])})

# Debug: check labels before training
print("Train label sample:", train_dataset["label"][:5])
print("Test label sample:", test_dataset["label"][:5])
print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))

# Fix: set problem type to single label classification
model.config.problem_type = "single_label_classification"

trainer = Trainer(
    model=model,                    # pretrained model
    args=training_args,             # training settings
    train_dataset=train_dataset,    # 80% data for training
    eval_dataset=test_dataset,      # 20% data for evaluation
    processing_class=tokenizer,     # tokenizer for text processing
    compute_metrics=compute_metrics # function to calculate accuracy, f1, etc.
)

# Start training
trainer.train()

print("Model Training Completed")


Filter:   0%|          | 0/30405 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7602 [00:00<?, ? examples/s]

Map:   0%|          | 0/30405 [00:00<?, ? examples/s]

Map:   0%|          | 0/7602 [00:00<?, ? examples/s]

Train label sample: [0, 2, 0, 0, 0]
Test label sample: [2, 2, 1, 2, 1]
Train dataset size: 30405
Test dataset size: 7602


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.723761,0.673707,0.716522,0.721288,0.716522,0.714320


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Training Completed


what does [1872/7602  07:50 < 21:00, 4.31  it/s, Epoch 0.62/2] means in  trainning.
Here is what each part means:
1872/7602 — completed 1872 steps out of 7602 total training steps
07:50 — time elapsed so far (7 minutes 50 seconds)
21:00 — estimated time remaining (21 minutes)
4.31 it/s — processing 4.31 iterations (batches) per second
Epoch 0.62/2 — currently 62% through the first epoch out of 2 total epochs. So it hasn't even finished Epoch 1 yet.
In simple terms — your model has completed about 25% of the total training and still has about 21 minutes left.

An **epoch** means the model has gone through the **entire training dataset once**.

In our case we set `num_train_epochs=2` which means:

- **Epoch 1** — the model reads and learns from all NaijaSenti tweets once
- **Epoch 2** — the model reads and learns from all NaijaSenti tweets again

**Why go through the data more than once?**
One pass is usually not enough for the model to learn well. Each time it goes through the data it gets a little better at predicting sentiments, like how you understand a textbook better the second time you read it.

**Why not do 100 epochs?**
Too many epochs causes **overfitting** — where the model memorizes the training data instead of learning general patterns, and performs poorly on new unseen data. 2-3 epochs is usually a good starting point for fine-tuning pretrained models.

when epoch is 1
Training samples = 30405
Batch size       = 16
Steps per epoch  = 30405 ÷ 16 = 1901 steps
Total steps      = 1901 × 1 epoch = 1901

In [10]:
# Step 10: Save the fine-tuned model and tokenizer
# Save the fine-tuned model and tokenizer
trainer.save_model("./naijasenti_model")
tokenizer.save_pretrained("./naijasenti_model")

print("Fine-tuned Model Saved Successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned Model Saved Successfully


In [9]:
# STEP 9: FINE-TUNING
# Fine-tuning adjusts the pretrained model to better understand Nigerian language sentiments

print("Starting Fine-Tuning...")

trainer.train() # train the model on the NaijaSenti dataset

print("Fine-Tuning Completed")


Starting Fine-Tuning...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.612462,0.631388,0.737438,0.739790,0.737438,0.736718


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-Tuning Completed


In [17]:
%%writefile app.py
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from flask import Flask, request, jsonify, render_template
import os

# Define the path where the model and tokenizer are saved
model_path = "./naijasenti_model"

# Initialize Flask app
app = Flask(__name__)

# Load model and tokenizer once when the app starts
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
print("Model loaded successfully!")

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    # Map numerical predictions back to sentiment labels
    sentiment_map = {0: "negative", 1: "neutral", 2: "positive"}
    return sentiment_map[predictions.item()]

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    text_to_analyze = data.get('text', '')

    if not text_to_analyze:
        return jsonify({'error': 'No text provided for analysis'}), 400

    sentiment = predict_sentiment(text_to_analyze)
    return jsonify({'text': text_to_analyze, 'sentiment': sentiment})

if __name__ == '__main__':
    # To run the app, you will execute this cell.
    # In a local environment, you'd typically run `python app.py` from your terminal.
    # For Colab, we need to expose it, for which you'd use ngrok or similar.
    # For simplicity, we'll run it directly, but access might be limited without tunneling.
    port = int(os.environ.get('PORT', 5000))
    app.run(host='0.0.0.0', port=port)


Overwriting app.py


In [16]:
import os

# Create the 'templates' directory if it doesn't exist
if not os.path.exists('templates'):
    os.makedirs('templates')
    print("Created directory: templates/")

# Write content to templates/index.html
with open('templates/index.html', 'w') as f:
    f.write("""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Multilingual Sentiment Analysis for Nigerian Tweets</title>
    <link href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;700&display=swap" rel="stylesheet">
    <style>
        body {
            font-family: 'Roboto', sans-serif;
            background: linear-gradient(to right, #4CAF50, #8BC34A); /* Green gradient */
            color: #333;
            margin: 0;
            padding: 0;
            display: flex;
            flex-direction: column;
            min-height: 100vh;
        }
        .container {
            max-width: 900px;
            margin: 40px auto;
            padding: 30px;
            background-color: #fff;
            border-radius: 12px;
            box-shadow: 0 6px 20px rgba(0, 0, 0, 0.1);
            flex-grow: 1;
            display: flex;
            flex-direction: column;
            gap: 25px;
        }
        header {
            text-align: center;
            margin-bottom: 20px;
            color: #00796B; /* Darker green for header */
        }
        header h1 {
            font-size: 2.5em;
            margin-bottom: 10px;
            color: #2E7D32;
        }
        header p {
            font-size: 1.1em;
            color: #555;
        }
        .section-title {
            font-size: 1.8em;
            color: #4CAF50;
            border-bottom: 2px solid #8BC34A;
            padding-bottom: 10px;
            margin-bottom: 20px;
        }
        .sentiment-stats {
            display: flex;
            justify-content: space-around;
            flex-wrap: wrap;
            margin-bottom: 30px;
        }
        .stat-card {
            background-color: #e8f5e9; /* Light green background */
            border: 1px solid #c8e6c9;
            border-radius: 8px;
            padding: 15px 20px;
            margin: 10px;
            text-align: center;
            width: 180px;
            box-shadow: 0 2px 8px rgba(0, 0, 0, 0.05);
        }
        .stat-card h3 {
            margin-top: 0;
            color: #388E3C;
        }
        .stat-card p {
            font-size: 1.2em;
            font-weight: bold;
            color: #1B5E20;
        }
        .form-section {
            background-color: #f9fbe7;
            border: 1px solid #dce775;
            border-radius: 8px;
            padding: 25px;
            margin-bottom: 30px;
        }
        textarea {
            width: calc(100% - 20px);
            padding: 10px;
            margin-bottom: 15px;
            border: 1px solid #ccc;
            border-radius: 5px;
            font-size: 1em;
            resize: vertical;
            min-height: 100px;
        }
        button {
            background-color: #4CAF50; /* Green button */
            color: white;
            padding: 12px 25px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-size: 1.1em;
            transition: background-color 0.3s ease;
        }
        button:hover {
            background-color: #689F38; /* Darker green on hover */
        }
        #result {
            margin-top: 20px;
            padding: 15px;
            border: 1px solid #dcdcdc;
            border-radius: 8px;
            background-color: #f5f5f5;
            font-size: 1.1em;
            display: none; /* Hidden by default */
        }
        #result.positive {
            background-color: #e8f5e9;
            border-color: #a5d6a7;
            color: #2E7D32;
        }
        #result.negative {
            background-color: #ffebee;
            border-color: #ef9a9a;
            color: #C62828;
        }
        #result.neutral {
            background-color: #fffde7;
            border-color: #ffe082;
            color: #FF8F00;
        }
        footer {
            margin-top: auto;
            text-align: center;
            padding: 20px;
            color: #f5f5f5;
            background-color: #2E7D32; /* Dark green footer */
            font-size: 0.9em;
            box-shadow: 0 -2px 10px rgba(0, 0, 0, 0.1);
        }
    </style>
</head>
<body>
    <div class="container">
        <header>
            <h1>Multilingual Sentiment Analysis for Nigerian Tweets</h1>
            <p>Leveraging NaijaSenti: Multilingual Dataset (Hausa, Igbo, Pidgin, Yoruba)</p>
        </header>

        <div class="section-title">Sentiment Distribution in Training Data</div>
        <div class="sentiment-stats">
            <div class="stat-card">
                <h3>Negative</h3>
                <p>{{ stats.percentages['0'] if stats else 'N/A' }}%</p>
                <p>({{ stats.counts['0'] if stats else 'N/A' }} samples)</p>
            </div>
            <div class="stat-card">
                <h3>Neutral</h3>
                <p>{{ stats.percentages['1'] if stats else 'N/A' }}%</p>
                <p>({{ stats.counts['1'] if stats else 'N/A' }} samples)</p>
            </div>
            <div class="stat-card">
                <h3>Positive</h3>
                <p>{{ stats.percentages['2'] if stats else 'N/A' }}%</p>
                <p>({{ stats.counts['2'] if stats else 'N/A' }} samples)</p>
            </div>
            <div class="stat-card">
                <h3>Total Samples</h3>
                <p>{{ stats.total if stats else 'N/A' }}</p>
            </div>
        </div>

        <div class="section-title">Predict Sentiment</div>
        <div class="form-section">
            <form id="sentimentForm">
                <textarea id="text_input" placeholder="Enter your tweet or text here..." rows="5"></textarea><br>
                <button type="submit">Analyze Sentiment</button>
            </form>
            <div id="result">
                Analyzed Text: <span id="analyzed_text"></span><br>
                Predicted Sentiment: <span id="predicted_sentiment"></span>
            </div>
        </div>
    </div>

    <footer>
        Iconic University Capstone Project
    </footer>

    <script>
        document.getElementById('sentimentForm').addEventListener('submit', async function(event) {
            event.preventDefault();
            const textInput = document.getElementById('text_input').value;
            const resultDiv = document.getElementById('result');
            const analyzedTextSpan = document.getElementById('analyzed_text');
            const predictedSentimentSpan = document.getElementById('predicted_sentiment');

            resultDiv.style.display = 'none'; // Hide previous result
            resultDiv.className = ''; // Clear previous sentiment class

            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json'
                    },
                    body: JSON.stringify({ text: textInput })
                });
                const data = await response.json();

                if (response.ok) {
                    analyzedTextSpan.textContent = data.text;
                    predictedSentimentSpan.textContent = data.sentiment;
                    resultDiv.classList.add(data.sentiment); // Add class for styling
                    resultDiv.style.display = 'block'; // Show the result
                } else {
                    alert('Error: ' + data.error);
                }
            } catch (error) {
                console.error('Fetch error:', error);
                alert('An error occurred while connecting to the server.');
            }
        });
    </script>
</body>
</html>""")
print("Created file: templates/index.html")


Created file: templates/index.html


In [ ]:
!python app.py

Loading model...
Loading weights: 100% 201/201 [00:00<00:00, 904.15it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
Model loaded successfully!
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit


In [14]:
import requests
import json

# The internal IP address Colab uses to run local servers
# Make sure the 'app.py' cell is still running
FLASK_APP_URL = "http://172.28.0.12:5000/predict"

headers = {'Content-Type': 'application/json'}

# Test with a positive sentence
text_positive = "This product is amazing! I love it."
response_positive = requests.post(FLASK_APP_URL, data=json.dumps({'text': text_positive}), headers=headers)
print(f"Text: '{text_positive}'")
print(f"Prediction: {response_positive.json()}")

# Test with a negative sentence
text_negative = "This service was terrible, very disappointed."
response_negative = requests.post(FLASK_APP_URL, data=json.dumps({'text': text_negative}), headers=headers)
print(f"\nText: '{text_negative}'")
print(f"Prediction: {response_negative.json()}")

# Test with a neutral sentence
text_neutral = "The weather is neither good nor bad today."
response_neutral = requests.post(FLASK_APP_URL, data=json.dumps({'text': text_neutral}), headers=headers)
print(f"\nText: '{text_neutral}'")
print(f"Prediction: {response_neutral.json()}")


ConnectionError: HTTPConnectionPool(host='172.28.0.12', port=5000): Max retries exceeded with url: /predict (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x780eb51cbce0>: Failed to establish a new connection: [Errno 111] Connection refused'))

If you want a more visual way, we could also modify the `templates/index.html` file to include a simple form where you can type text and see the sentiment result directly in your browser. Would you like me to create that for you?

# Task
Calculate the frequency and percentage of each sentiment label from the `data` DataFrame by modifying `app.py` to load the dataset and compute these statistics within the `/` route, then pass them to the `index.html` template.

## Calculate Sentiment Statistics

### Subtask:
Modify `app.py` to calculate the frequency and percentage of each sentiment label from the `data` DataFrame. This will prepare the statistics to be displayed on the web page.


**Reasoning**:
To calculate sentiment statistics within `app.py`, I need to incorporate the necessary data loading and preprocessing steps from the previous notebook cells. This includes importing `pandas`, `zipfile`, and `os`, cloning and extracting the dataset, loading the language-specific TSV files, concatenating them, mapping labels, and cleaning the data. After preprocessing, I will calculate the frequency and percentage of each sentiment label and store them in a dictionary. I will then write the entire modified `app.py` content back to the file using `%%writefile`, ensuring the new code is integrated logically after the model loading and before the Flask app's main execution block.



In [ ]:
%%writefile app.py
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from flask import Flask, request, jsonify, render_template
import os
import pandas as pd, zipfile # Added pandas, zipfile

# Define the path where the model and tokenizer are saved
model_path = "./naijasenti_model"

# Initialize Flask app
app = Flask(__name__)

# Load model and tokenizer once when the app starts
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
print("Model loaded successfully!")

# --- Start of new code for sentiment statistics ---
print("Loading data for sentiment statistics...")
# Clone the NaijaSenti repository
if not os.path.exists("NaijaSenti"):
    print("Cloning NaijaSenti repository...")
    os.system("git clone https://github.com/hausanlp/NaijaSenti.git")

# Extract the data.zip file if the annotated_tweets directory doesn't exist
if not os.path.exists("NaijaSenti/data/annotated_tweets/hau/train.tsv"):
    print("Extracting data.zip...")
    with zipfile.ZipFile("NaijaSenti/data.zip", 'r') as zip_ref:
        zip_ref.extractall("NaijaSenti")

# Load all 4 languages and combine them
languages = ["hau", "ibo", "pcm", "yor"]
all_data = []

for lang in languages:
    lang_path = f"NaijaSenti/data/annotated_tweets/{lang}/train.tsv"
    if os.path.exists(lang_path):
        df_lang = pd.read_csv(lang_path, sep='\t')
        all_data.append(df_lang)
    else:
        print(f"Warning: {lang_path} not found. Skipping {lang} data.")

sentiment_stats = {}
if all_data:
    data_df = pd.concat(all_data, ignore_index=True)

    # Keep only text and label columns
    data_df = data_df[["tweet", "label"]].rename(columns={"tweet": "text"})

    # Convert labels to numbers: negative=0, neutral=1, positive=2
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    data_df["label"] = data_df["label"].map(label_map)

    # Remove rows with NaN in 'text' or 'label'
    data_df = data_df.dropna(subset=["text", "label"])
    data_df["label"] = data_df["label"].astype(int) # Convert label to integer after dropna

    # Ensure 'text' column is string type
    data_df["text"] = data_df["text"].astype(str)

    # Calculate sentiment statistics
    sentiment_counts = data_df["label"].value_counts().sort_index()
    total_samples = sentiment_counts.sum()
    sentiment_percentages = (sentiment_counts / total_samples * 100).round(2)

    # Prepare sentiment_stats for display, ensuring keys are strings for JSON
    sentiment_stats = {
        "counts": {str(k): int(v) for k, v in sentiment_counts.items()},
        "percentages": {str(k): float(v) for k, v in sentiment_percentages.items()},
        "total": int(total_samples)
    }
    print(f"Sentiment statistics calculated: {sentiment_stats}")
else:
    print("No data loaded for sentiment statistics. Defaulting to empty stats.")
    sentiment_stats = {"counts": {"0": 0, "1": 0, "2": 0}, "percentages": {"0": 0.0, "1": 0.0, "2": 0.0}, "total": 0}

# Add a route to serve the statistics
@app.route('/stats')
def stats():
    return jsonify(sentiment_stats)
# --- End of new code for sentiment statistics ---

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    # Map numerical predictions back to sentiment labels
    sentiment_map = {0: "negative", 1: "neutral", 2: "positive"}
    return sentiment_map[predictions.item()]

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    text_to_analyze = data.get('text', '')

    if not text_to_analyze:
        return jsonify({'error': 'No text provided for analysis'}), 400

    sentiment = predict_sentiment(text_to_analyze)
    return jsonify({'text': text_to_analyze, 'sentiment': sentiment})

if __name__ == '__main__':
    port = int(os.environ.get('PORT', 5000))
    app.run(host='0.0.0.0', port=port)


## Update Flask App to Pass Statistics and Handle Form

### Subtask:
Adjust the `/` route in `app.py` to pass the calculated sentiment statistics to the `index.html` template. Also, ensure the `/predict` route is correctly integrated with the new form structure expected from `index.html`.


**Reasoning**:
I need to modify the `app.py` file to pass sentiment statistics to the `index.html` template and update the `/predict` route to handle form data. I will rewrite the entire `app.py` file using `%%writefile` with these changes.



In [15]:
%%writefile app.py
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from flask import Flask, request, jsonify, render_template
import os
import pandas as pd, zipfile # Added pandas, zipfile

# Define the path where the model and tokenizer are saved
model_path = "./naijasenti_model"

# Initialize Flask app
app = Flask(__name__)

# Load model and tokenizer once when the app starts
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
print("Model loaded successfully!")

# --- Start of new code for sentiment statistics ---
print("Loading data for sentiment statistics...")
# Clone the NaijaSenti repository
if not os.path.exists("NaijaSenti"):
    print("Cloning NaijaSenti repository...")
    os.system("git clone https://github.com/hausanlp/NaijaSenti.git")

# Extract the data.zip file if the annotated_tweets directory doesn't exist
if not os.path.exists("NaijaSenti/data/annotated_tweets/hau/train.tsv"):
    print("Extracting data.zip...")
    with zipfile.ZipFile("NaijaSenti/data.zip", 'r') as zip_ref:
        zip_ref.extractall("NaijaSenti")

# Load all 4 languages and combine them
languages = ["hau", "ibo", "pcm", "yor"]
all_data = []

for lang in languages:
    lang_path = f"NaijaSenti/data/annotated_tweets/{lang}/train.tsv"
    if os.path.exists(lang_path):
        df_lang = pd.read_csv(lang_path, sep='\t')
        all_data.append(df_lang)
    else:
        print(f"Warning: {lang_path} not found. Skipping {lang} data.")

sentiment_stats = {}
if all_data:
    data_df = pd.concat(all_data, ignore_index=True)

    # Keep only text and label columns
    data_df = data_df[["tweet", "label"]].rename(columns={"tweet": "text"})

    # Convert labels to numbers: negative=0, neutral=1, positive=2
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    data_df["label"] = data_df["label"].map(label_map)

    # Remove rows with NaN in 'text' or 'label'
    data_df = data_df.dropna(subset=["text", "label"])
    data_df["label"] = data_df["label"].astype(int) # Convert label to integer after dropna

    # Ensure 'text' column is string type
    data_df["text"] = data_df["text"].astype(str)

    # Calculate sentiment statistics
    sentiment_counts = data_df["label"].value_counts().sort_index()
    total_samples = sentiment_counts.sum()
    sentiment_percentages = (sentiment_counts / total_samples * 100).round(2)

    # Prepare sentiment_stats for display, ensuring keys are strings for JSON
    sentiment_stats = {
        "counts": {str(k): int(v) for k, v in sentiment_counts.items()},
        "percentages": {str(k): float(v) for k, v in sentiment_percentages.items()},
        "total": int(total_samples)
    }
    print(f"Sentiment statistics calculated: {sentiment_stats}")
else:
    print("No data loaded for sentiment statistics. Defaulting to empty stats.")
    sentiment_stats = {"counts": {"0": 0, "1": 0, "2": 0}, "percentages": {"0": 0.0, "1": 0.0, "2": 0.0}, "total": 0}

# Add a route to serve the statistics
@app.route('/stats')
def stats():
    return jsonify(sentiment_stats)
# --- End of new code for sentiment statistics ---

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    # Map numerical predictions back to sentiment labels
    sentiment_map = {0: "negative", 1: "neutral", 2: "positive"}
    return sentiment_map[predictions.item()]

@app.route('/')
def home():
    # Pass sentiment_stats to the index.html template
    return render_template('index.html', stats=sentiment_stats)

@app.route('/predict', methods=['POST'])
def predict():
    # Retrieve text input from form submission
    text_to_analyze = request.form.get('text', '')

    if not text_to_analyze:
        return jsonify({'error': 'No text provided for analysis'}), 400

    sentiment = predict_sentiment(text_to_analyze)
    return jsonify({'text': text_to_analyze, 'sentiment': sentiment})

if __name__ == '__main__':
    port = int(os.environ.get('PORT', 5000))
    app.run(host='0.0.0.0', port=port)


Overwriting app.py
